# ElasticNet (EAD) Interactive Challenge

This notebook turns the earlier ElasticNet theory into a reproducible challenge workflow. It is deliberately separate from `ElasticNet.ipynb`: the challenge supplies a different model and works through an HTTP API, while the teaching notebook trains its own local model.

The workflow is: fetch the fixed image and limits, reproduce the server model locally, optimize an adversarial image in `[0,1]` pixel space, simulate PNG encoding, verify every constraint, ask `/predict` to confirm the server result, and only then optionally call `/submit`.

> **Before running:** start the target on the HTB page and replace `INSTANCE_IP:PORT` in the configuration cell. Run the cells from top to bottom. The final submission switch defaults to `False`.

## 1. Imports and target configuration

The API transports images as base64-encoded PNG files. NumPy and Pillow handle image conversion, Requests talks to the challenge server, and PyTorch calculates gradients for the attack.

The attack always optimizes pixels in the server's `[0,1]` space. Normalization belongs inside the model wrapper; otherwise the distance limits would be measured in the wrong coordinate system.

In [ ]:
import base64
import io
from pathlib import Path

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

# Replace this placeholder after spawning the HTB target.
# Example shape only: BASE_URL = "http://10.10.10.10:12345"
BASE_URL = "http://INSTANCE_IP:PORT"

REQUEST_TIMEOUT = 20
OUTPUT_DIR = Path("output/challenge")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_PATH = OUTPUT_DIR / "elasticnet_weights.pth"
CANDIDATE_PATH = OUTPUT_DIR / "elasticnet_candidate.png"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(1337)
print(f"Using device: {DEVICE}")

if "INSTANCE_IP:PORT" in BASE_URL:
    print("Reminder: replace INSTANCE_IP:PORT before running the API cells.")

## 2. Image transport and metric helpers

Let the original image be $x$, the adversarial image be $x'$, and the perturbation be $\delta=x'-x$. Read that aloud as **“delta equals x-prime minus x.”**

The server checks three related quantities:

- $\lVert\delta\rVert_1=\sum_i|\delta_i|$: **“the L-one norm of delta equals the sum over i of the absolute value of delta sub i.”**
- $\lVert\delta\rVert_2=\sqrt{\sum_i\delta_i^2}$: **“the L-two norm of delta equals the square root of the sum over i of delta sub i squared.”**
- $D_{EN}=\lVert\delta\rVert_2^2+\beta\lVert\delta\rVert_1$: **“D sub E-N equals squared L-two plus beta times L-one.”**

A crucial distinction: `l2_max` limits ordinary L2, including the square root. The ElasticNet expression uses squared L2. The helper returns both names explicitly so we cannot confuse them.

PNG is an 8-bit format. Encoding rounds each pixel to one of 256 levels, so we must measure and classify the decoded PNG—not only the unrounded tensor produced by the optimizer.

In [ ]:
def x01_from_b64_png(encoded: str) -> np.ndarray:
    """Decode a base64 grayscale PNG into a float32 (28, 28) array in [0,1]."""
    raw = base64.b64decode(encoded)
    image = Image.open(io.BytesIO(raw)).convert("L")
    if image.size != (28, 28):
        raise ValueError(f"Expected a 28x28 PNG, received {image.size}")
    return np.asarray(image, dtype=np.float32) / 255.0


def b64_png_from_x01(image_x01: np.ndarray) -> str:
    """Clip, quantize, and encode a [0,1] array as a base64 grayscale PNG."""
    image_u8 = np.clip(np.rint(image_x01 * 255.0), 0, 255).astype(np.uint8)
    buffer = io.BytesIO()
    Image.fromarray(image_u8, mode="L").save(buffer, format="PNG", optimize=True)
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def png_round_trip(image_x01: np.ndarray) -> tuple[np.ndarray, str]:
    """Return exactly the pixels the server will decode, plus their base64 PNG."""
    encoded = b64_png_from_x01(image_x01)
    return x01_from_b64_png(encoded), encoded


def compute_challenge_distances(candidate: np.ndarray, original: np.ndarray, beta: float) -> dict:
    """Compute challenge metrics in [0,1] pixel space."""
    delta = candidate.astype(np.float64) - original.astype(np.float64)
    l1 = float(np.sum(np.abs(delta)))
    l2_squared = float(np.sum(delta ** 2))
    l2 = float(np.sqrt(l2_squared))
    linf = float(np.max(np.abs(delta)))
    elastic = l2_squared + beta * l1
    return {
        "l1": l1, "l2": l2, "l2_squared": l2_squared,
        "linf": linf, "elastic": elastic,
    }

## 3. Fetch the fixed challenge

`GET /challenge` returns the original image, its correct label, $\beta$ (pronounced **“beta”**), and three maximum distances. `GET /health` is checked first so a typo or expired instance fails with a clear message.

This challenge is deterministic: repeated fetches from the same instance should describe the same sample.

In [ ]:
if "INSTANCE_IP:PORT" in BASE_URL:
    raise RuntimeError("Replace INSTANCE_IP:PORT in the configuration cell, then rerun from the top.")

health_response = requests.get(f"{BASE_URL}/health", timeout=REQUEST_TIMEOUT)
health_response.raise_for_status()
print("Health:", health_response.json())

challenge_response = requests.get(f"{BASE_URL}/challenge", timeout=REQUEST_TIMEOUT)
challenge_response.raise_for_status()
challenge = challenge_response.json()

original_x01 = x01_from_b64_png(challenge["image_b64"])
true_label = int(challenge["label"])
beta = float(challenge["beta"])
elastic_max = float(challenge["elastic_max"])
l2_max = float(challenge["l2_max"])
l1_max = float(challenge["l1_max"])

print({
    "sample_index": challenge.get("sample_index"),
    "label": true_label,
    "beta": beta,
    "elastic_max": elastic_max,
    "l2_max": l2_max,
    "l1_max": l1_max,
})

## 4. Reproduce the server classifier locally

We need the local model because `/predict` returns only a class and confidence; it cannot provide the gradient of the loss with respect to every input pixel. Downloading `/weights` lets PyTorch compute those gradients locally.

The server first normalizes a pixel $p$ as

$$p_{norm}=\frac{p-\mu}{\sigma}$$

Read aloud: **“p sub norm equals p minus mu, divided by sigma.”** Here $\mu$ is pronounced **“mew”** and means the MNIST mean; $\sigma$ is pronounced **“sigma”** and means the MNIST standard deviation.

`NormalizedChallengeModel` keeps that transformation inside `forward()`. Therefore FISTA still manipulates legal `[0,1]` pixels, while the inner classifier sees the normalized values on which it was trained.

In [ ]:
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081


class SimpleClassifier(nn.Module):
    """Architecture used by the challenge server."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        return self.fc2(x)  # Raw per-class scores, called logits.


class NormalizedChallengeModel(nn.Module):
    """Accept [0,1] pixels and apply MNIST normalization before classification."""

    def __init__(self, classifier: nn.Module):
        super().__init__()
        self.classifier = classifier

    def forward(self, x01: torch.Tensor) -> torch.Tensor:
        return self.classifier((x01 - MNIST_MEAN) / MNIST_STD)


weights_response = requests.get(f"{BASE_URL}/weights", timeout=REQUEST_TIMEOUT)
weights_response.raise_for_status()
WEIGHTS_PATH.write_bytes(weights_response.content)

classifier = SimpleClassifier().to(DEVICE)
state_dict = torch.load(WEIGHTS_PATH, map_location=DEVICE)
classifier.load_state_dict(state_dict)
classifier.eval()  # Disables dropout so inference and gradients are deterministic.
model = NormalizedChallengeModel(classifier).to(DEVICE).eval()

original = torch.from_numpy(original_x01).unsqueeze(0).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    local_clean_pred = int(model(original).argmax(dim=1).item())

server_clean = requests.post(
    f"{BASE_URL}/predict",
    json={"image_b64": challenge["image_b64"]},
    timeout=REQUEST_TIMEOUT,
).json()
print({"ground_truth": true_label, "local_prediction": local_clean_pred,
       "server_prediction": server_clean["pred"]})
assert local_clean_pred == int(server_clean["pred"]) == true_label, (
    "Local/server predictions disagree. Check the architecture and normalization."
)

## 5. ElasticNet building blocks

For an untargeted attack, the adversarial hinge loss is

$$f(x',y)=\max\left(Z_y(x')-\max_{j\ne y}Z_j(x')+\kappa,0\right).$$

Read aloud: **“f of x-prime and y equals the maximum of: Z sub y of x-prime minus the maximum Z sub j of x-prime over j not equal to y, plus kappa; and zero.”** It becomes zero once some incorrect class leads the true class by the required confidence margin $\kappa$ (pronounced **“kappa”**).

The smooth objective is $c f(x',y)+\lVert x'-x\rVert_2^2$. Read $c$ as **“see.”** Gradient descent handles these differentiable terms. The nonsmooth $\beta\lVert x'-x\rVert_1$ term is handled separately by the proximal operator.

Soft thresholding acts on the perturbation, not the image. For threshold $\tau$, values with magnitude at most $\tau$ become exactly zero; larger values move toward zero by $\tau$. That exact-zero behavior creates sparsity.

In [ ]:
def adversarial_hinge_loss(logits: torch.Tensor, labels: torch.Tensor, kappa: float) -> torch.Tensor:
    """Per-example untargeted logit-margin loss."""
    true_logits = logits.gather(1, labels[:, None]).squeeze(1)
    competitor_logits = logits.masked_fill(
        F.one_hot(labels, num_classes=logits.shape[1]).bool(), float("-inf")
    ).max(dim=1).values
    return torch.clamp(true_logits - competitor_logits + kappa, min=0.0)


def soft_threshold_perturbation(candidate: torch.Tensor, original: torch.Tensor, threshold: float) -> torch.Tensor:
    """Apply the L1 proximal operator to candidate-original."""
    delta = candidate - original
    sparse_delta = torch.sign(delta) * torch.clamp(torch.abs(delta) - threshold, min=0.0)
    return torch.clamp(original + sparse_delta, 0.0, 1.0)


def torch_distances(candidate: torch.Tensor, original: torch.Tensor, beta_value: float):
    """Return per-example L1, squared L2, ordinary L2, and ElasticNet distance."""
    delta = candidate - original
    l1 = delta.abs().sum(dim=(1, 2, 3))
    l2_squared = delta.square().sum(dim=(1, 2, 3))
    l2 = torch.sqrt(l2_squared)
    elastic = l2_squared + beta_value * l1
    return l1, l2_squared, l2, elastic


def is_misclassified(candidate: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        return model(candidate).argmax(dim=1).ne(labels)

## 6. FISTA and binary search

ISTA performs a gradient step followed by soft thresholding. FISTA adds an extrapolated point carrying some recent movement forward. At iteration $t$, its common coefficient is

$$\frac{t-1}{t+2}.$$

Read aloud: **“t minus one, divided by t plus two.”** At the first iteration it is zero; later it approaches one.

The outer binary search tunes $c$, the importance of achieving misclassification versus remaining close to the original. Successful trials lower the upper bound on $c$; failed trials raise its lower bound. Within every trial, we retain only candidates that misclassify and satisfy **all three challenge constraints**.

The default values favor reliability over speed. If the CPU run is slow, reduce `MAX_ITERATIONS` while experimenting, then restore it for the final candidate.

In [ ]:
BINARY_SEARCH_STEPS = 9
MAX_ITERATIONS = 1200
LEARNING_RATE = 0.01
INITIAL_C = 0.01
KAPPA = 0.0


def run_elasticnet_attack(original: torch.Tensor, label: torch.Tensor):
    batch_size = original.shape[0]
    lower_c = torch.zeros(batch_size, device=DEVICE)
    upper_c = torch.full((batch_size,), 1e10, device=DEVICE)
    c = torch.full((batch_size,), INITIAL_C, device=DEVICE)

    best_candidate = original.detach().clone()
    best_elastic = torch.full((batch_size,), float("inf"), device=DEVICE)
    best_found = torch.zeros(batch_size, dtype=torch.bool, device=DEVICE)

    for search_step in range(BINARY_SEARCH_STEPS):
        x_k = original.detach().clone()
        y_k = x_k.clone()

        for iteration in range(MAX_ITERATIONS):
            y_var = y_k.detach().requires_grad_(True)
            logits = model(y_var)
            hinge = adversarial_hinge_loss(logits, label, KAPPA)
            squared_l2 = (y_var - original).square().sum(dim=(1, 2, 3))
            smooth_loss = (c * hinge + squared_l2).sum()
            gradient = torch.autograd.grad(smooth_loss, y_var)[0]

            with torch.no_grad():
                gradient_candidate = y_var - LEARNING_RATE * gradient
                x_next = soft_threshold_perturbation(
                    gradient_candidate, original, LEARNING_RATE * beta
                )
                momentum = iteration / (iteration + 3.0)
                y_next = torch.clamp(x_next + momentum * (x_next - x_k), 0.0, 1.0)
                x_k, y_k = x_next, y_next

                # Check intermediate iterates; the final iterate is not always the best one.
                if iteration % 10 == 0 or iteration == MAX_ITERATIONS - 1:
                    # The API receives 8-bit PNG pixels, so rank the quantized iterate.
                    quantized_x = torch.round(x_k * 255.0) / 255.0
                    success = is_misclassified(quantized_x, label)
                    l1, _, l2, elastic = torch_distances(quantized_x, original, beta)
                    feasible = success & (elastic <= elastic_max) & (l2 <= l2_max) & (l1 <= l1_max)
                    improved = feasible & (elastic < best_elastic)
                    best_candidate[improved] = quantized_x[improved]
                    best_elastic[improved] = elastic[improved]
                    best_found |= feasible

        trial_success = is_misclassified(x_k, label)
        upper_c = torch.where(trial_success, torch.minimum(upper_c, c), upper_c)
        lower_c = torch.where(trial_success, lower_c, torch.maximum(lower_c, c))
        finite_upper = upper_c < 1e9
        c = torch.where(finite_upper, (lower_c + upper_c) / 2, c * 10)
        print(f"Search {search_step + 1}/{BINARY_SEARCH_STEPS}: "
              f"c={c.item():.6g}, misclassified={bool(trial_success.item())}, "
              f"feasible candidate found={bool(best_found.item())}")

    if not bool(best_found.all()):
        raise RuntimeError(
            "No locally feasible candidate was found. Inspect the printed search behavior and tune "
            "INITIAL_C, LEARNING_RATE, MAX_ITERATIONS, or KAPPA."
        )
    return best_candidate.detach()


label_tensor = torch.tensor([true_label], dtype=torch.long, device=DEVICE)
adversarial = run_elasticnet_attack(original, label_tensor)

## 7. Validate the actual PNG candidate

The optimizer returns floating-point pixels, but the API receives a PNG. This cell performs the encode/decode round trip, saves exactly those quantized pixels, and repeats prediction and metric checks.

Each printed margin is `maximum - observed`. A positive margin means the candidate is under that limit; a negative margin means it exceeds the constraint. Keeping some positive headroom is safer than landing exactly on a boundary.

In [ ]:
candidate_float = adversarial[0, 0].detach().cpu().numpy()
candidate_png, candidate_b64 = png_round_trip(candidate_float)
Image.fromarray(np.rint(candidate_png * 255).astype(np.uint8), mode="L").save(CANDIDATE_PATH)

metrics = compute_challenge_distances(candidate_png, original_x01, beta)
candidate_tensor = torch.from_numpy(candidate_png).unsqueeze(0).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    local_candidate_pred = int(model(candidate_tensor).argmax(dim=1).item())

checks = {
    "misclassified": local_candidate_pred != true_label,
    "pixels_in_range": bool(np.all((candidate_png >= 0) & (candidate_png <= 1))),
    "elastic_within_limit": metrics["elastic"] <= elastic_max,
    "l2_within_limit": metrics["l2"] <= l2_max,
    "l1_within_limit": metrics["l1"] <= l1_max,
}
margins = {
    "elastic_margin": elastic_max - metrics["elastic"],
    "l2_margin": l2_max - metrics["l2"],
    "l1_margin": l1_max - metrics["l1"],
}
print("Local PNG prediction:", local_candidate_pred, "(true label:", true_label, ")")
print("Metrics:", metrics)
print("Margins:", margins)
print("Checks:", checks)
print("Saved candidate:", CANDIDATE_PATH.resolve())
assert all(checks.values()), "PNG quantization broke at least one local requirement."

## 8. Confirm with `/predict`

This is a non-submitting integration check. It verifies that the server interprets our base64 PNG the same way as the local model. It cannot award the flag.

In [ ]:
predict_response = requests.post(
    f"{BASE_URL}/predict",
    json={"image_b64": candidate_b64},
    timeout=REQUEST_TIMEOUT,
)
predict_response.raise_for_status()
server_candidate = predict_response.json()
print("Server candidate result:", server_candidate)
assert int(server_candidate["pred"]) == local_candidate_pred
assert int(server_candidate["pred"]) != true_label

## 9. Optional final submission

Submitting asks the server to enforce every constraint and return the flag. It is intentionally isolated behind `SUBMIT_TO_SERVER = False`. Review the local checks, positive margins, saved PNG, and `/predict` result first. Then change the switch to `True` and run only this cell.

In [ ]:
SUBMIT_TO_SERVER = False

if not SUBMIT_TO_SERVER:
    print("Submission skipped. Set SUBMIT_TO_SERVER=True after reviewing every check.")
else:
    submit_response = requests.post(
        f"{BASE_URL}/submit",
        json={"image_b64": candidate_b64},
        timeout=REQUEST_TIMEOUT,
    )
    print("HTTP status:", submit_response.status_code)
    print("Server response:", submit_response.json())
    submit_response.raise_for_status()

## Troubleshooting guide

- **Health request fails:** verify that the HTB instance is running and that `BASE_URL` includes both IP and port. Instances expire.
- **Local prediction differs from the server:** do not attack yet. Recheck the downloaded weights, exact architecture, `eval()` mode, PNG decoding, and MNIST normalization.
- **The loss never reaches misclassification:** increase `INITIAL_C`, add binary-search steps, or reduce the learning rate if optimization oscillates.
- **Misclassified but over a distance bound:** allow binary search more iterations, reduce `KAPPA`, or tune the learning rate. A stronger attack is not automatically a valid attack.
- **Float candidate passes but PNG candidate fails:** optimization landed too close to a boundary. Seek more margin; never submit the unverified float result.
- **Server rejects a locally valid result:** compare the server-returned metrics with `compute_challenge_distances`. Confirm ordinary L2 was not confused with squared L2.